# StressID and Experiment Dataset comparison

In [1]:
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

STRESSID_DATA_PATH = "../../.."
STRESSID_LABELS_SEPARATOR = ","
STRESSID_LABELS_FILENAME = f"{STRESSID_DATA_PATH}/stressID/labels.csv"
STRESSID_FEATURES_SEPARATOR = ","
STRESSID_FEATURES_DIRECTORY = f"{STRESSID_DATA_PATH}/Reprod-Features-StdSubject"
STRESSID_FEATURES_FILENAME = f"{STRESSID_FEATURES_DIRECTORY}/ecg_eda_features.csv"
STRESSID_RAWDATA_PATH = f"{STRESSID_DATA_PATH}/stressid-data"

EXPDATA_DATA_PATH = "../../../experiment-data"
EXPDATA_LABELS_SEPARATOR = ","
EXPDATA_LABELS_FILENAME = f"{EXPDATA_DATA_PATH}/labels.csv"
EXPDATA_FEATURES_SEPARATOR = ";"
EXPDATA_FEATURES_DIRECTORY = f"{EXPDATA_DATA_PATH}/extracted-features"
EXPDATA_FEATURES_FILENAME = f"{EXPDATA_FEATURES_DIRECTORY}/all_features.stdsubj.csv"

In [2]:
stressid_df = pd.read_csv(STRESSID_FEATURES_FILENAME, sep=STRESSID_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = stressid_df.shape
print("======================================================")
print(f"StressID dataset contains {n_rows} samples with {n_cols} features")
# display(stressid_df)

si_labels_df = pd.read_csv(STRESSID_LABELS_FILENAME, sep=STRESSID_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = si_labels_df.shape
print("======================================================")
print(f"StressID labels contains {n_rows} labels with {n_cols} types of classifications")
# display(si_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(stressid_df.merge(si_labels_df, left_index=True, right_index=True).index)
si_labels = si_labels_df.loc[idx]
si_X = stressid_df.loc[idx]
n_rows, n_cols = si_X.shape
print("======================================================")
print(f"StressID classification dataset contains {n_rows} samples with {n_cols} features")

si_video_tasks = ["Relax", "Video1", "Video2"]
si_video_tasks_mask = [task.split("_")[1] in si_video_tasks for task in si_X.index]

si_bclass_labels = si_labels["binary-stress"]
display(si_bclass_labels)
si_subject_to_group: dict[str, int] = {}
si_group_counter = 0
si_groups_list: list[int] = []
for col_name, label in si_bclass_labels.items():
    subject_id, task = col_name.split("_")
    if subject_id not in si_subject_to_group:
        si_subject_to_group[subject_id] = si_group_counter
        si_group_counter += 1
    si_groups_list.append(si_subject_to_group[subject_id])
# display(si_groups_list)


expdata_df = pd.read_csv(EXPDATA_FEATURES_FILENAME, sep=EXPDATA_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = expdata_df.shape
print("\n\n======================================================")
print(f"ExpData dataset contains {n_rows} samples with {n_cols} features")
# display(expdata_df)

ed_labels_df = pd.read_csv(EXPDATA_LABELS_FILENAME, sep=EXPDATA_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = ed_labels_df.shape
print("======================================================")
print(f"ExpData labels contains {n_rows} labels with {n_cols} types of classifications")
# display(ed_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(expdata_df.merge(ed_labels_df, left_index=True, right_index=True).index)
ed_labels = ed_labels_df.loc[idx]
ed_X = expdata_df.loc[idx]
n_rows, n_cols = ed_X.shape
print("======================================================")
print(f"ExpData classification dataset contains {n_rows} samples with {n_cols} features")

ed_bclass_labels = ed_labels["binary-stress"]
display(ed_bclass_labels)
ed_subject_to_group: dict[str, int] = {}
ed_group_counter = 0
ed_groups_list: list[int] = []
for col_name, label in ed_bclass_labels.items():
    subject_id, task = col_name.split("-")
    if subject_id not in ed_subject_to_group:
        ed_subject_to_group[subject_id] = ed_group_counter
        ed_group_counter += 1
    ed_groups_list.append(ed_subject_to_group[subject_id])
# display(ed_groups_list)

all_X = pd.concat([si_X, ed_X])
all_y = pd.concat([si_bclass_labels, ed_bclass_labels])
stress_tasks = ["Baseline", "AmusementClip", "StressClip", "EmoReset"]
frust_tasks = ["FormL", "FormM"]
ed_stress_task = ["ED-Video" for task in ed_bclass_labels.index if task.split("-")[1] in stress_tasks]
ed_frust_task = ["ED-Calc" for task in ed_bclass_labels.index if task.split("-")[1] in frust_tasks]
print(f"ExpData has {len(ed_stress_task)} stress tasks, and {len(ed_frust_task)} frustration tasks")
all_groups = np.concatenate([np.full((si_X.shape[0],), "OD-All"), ed_stress_task, ed_frust_task])
si_subject_list = pd.Series(si_groups_list)
ed_subject_list_offsetted = pd.Series(ed_groups_list) + len(si_subject_to_group)
all_subjects_list = pd.concat([si_subject_list, ed_subject_list_offsetted], ignore_index=True)
n_rows, n_cols = all_X.shape
print("======================================================")
print(f"Combined classification dataset contains {n_rows} samples with {n_cols} features, groups by dataset is {all_groups.size} members long")
print(f"Combined datasets subjects list is {len(all_subjects_list)} members long, having {len(set(all_subjects_list))} distinct subjects")

# Picking a small sample of all subjects to show
n_subjects_per_dataset = 3
sel_si_subjects = np.random.choice(range(len(si_subject_to_group)), size=n_subjects_per_dataset)
sel_ed_subjects = np.random.choice(range(len(ed_subject_to_group)), size=n_subjects_per_dataset)
si_subject_mask = [n in sel_si_subjects for n in si_groups_list]
ed_subject_mask = [n in sel_ed_subjects for n in ed_groups_list]
si_filtered_subjects_rows = pd.Series([str(subject) for subject, flag in zip(si_subject_list.array, si_subject_mask) if flag])
si_filtered_classes_rows = pd.Series([str(label) for label, flag in zip(si_bclass_labels.values, si_subject_mask) if flag])
edoff_filtered_subjects_rows = pd.Series([str(subject) for subject, flag in zip(ed_subject_list_offsetted.array, ed_subject_mask) if flag])
ed_filtered_classes_rows = pd.Series([str(label) for label, flag in zip(ed_bclass_labels.values, ed_subject_mask) if flag])

all_subsample_subjects_rows = pd.concat([si_filtered_subjects_rows, edoff_filtered_subjects_rows], ignore_index=True)
all_subsample_classes_rows = pd.concat([si_filtered_classes_rows, ed_filtered_classes_rows], ignore_index=True)

rel_X = pd.concat([si_X[si_video_tasks_mask], ed_X])
rel_y = pd.concat([si_bclass_labels[si_video_tasks_mask], ed_bclass_labels])
rel_groups = np.concatenate([np.full((si_X[si_video_tasks_mask].shape[0],), "OD-Video"), ed_stress_task, ed_frust_task])
rel_subjects_list = pd.concat([si_subject_list[si_video_tasks_mask], ed_subject_list_offsetted], ignore_index=True)
n_rows, n_cols = rel_X.shape
print("======================================================")
print(f"Relevant tasks combined dataset contains {n_rows} samples with {n_cols} features, groups by relevant tasks is {rel_groups.size} members long")
print(f"Combined relevant tasks subjects list is {len(rel_subjects_list)} members long, having {len(set(rel_subjects_list))} distinct subjects")

sisel_subject_mask = np.array(si_subject_mask)[si_video_tasks_mask]
sisel_filtered_subjects_rows = pd.Series([str(subject) for subject, flag in zip(si_subject_list[si_video_tasks_mask].array, sisel_subject_mask) if flag])
sisel_filtered_classes_rows = pd.Series([str(label) for label, flag in zip(si_bclass_labels[si_video_tasks_mask].array, sisel_subject_mask) if flag])
#print(f"Filtered subject list is {si_subject_list[si_video_tasks_mask].size} long")
#print(f"Subject mask list is {sisel_subject_mask.size} long")
#print(f"Subject IDs selected by tasks and subsampled is {sisel_filtered_subjects_rows.size} long")

sel_subsample_subjects_rows = pd.concat([sisel_filtered_subjects_rows, edoff_filtered_subjects_rows], ignore_index=True)
sel_subsample_classes_rows = pd.concat([sisel_filtered_classes_rows, ed_filtered_classes_rows], ignore_index=True)

StressID dataset contains 773 samples with 70 features
StressID labels contains 700 labels with 3 types of classifications
StressID classification dataset contains 699 samples with 70 features


subject/task
2ea4_Breathing    0
2ea4_Counting1    1
2ea4_Counting2    1
2ea4_Counting3    1
2ea4_Math         1
                 ..
y9z6_Relax        0
y9z6_Speaking     1
y9z6_Stroop       1
y9z6_Video1       1
y9z6_Video2       0
Name: binary-stress, Length: 699, dtype: int64



ExpData dataset contains 147 samples with 70 features
ExpData labels contains 126 labels with 3 types of classifications
ExpData classification dataset contains 126 samples with 70 features


subject/task
01-Baseline         0
01-AmusementClip    0
01-StressClip       1
01-EmoReset         0
01-FormL            1
                   ..
21-AmusementClip    0
21-StressClip       1
21-EmoReset         0
21-FormL            0
21-FormM            0
Name: binary-stress, Length: 126, dtype: int64

ExpData has 84 stress tasks, and 42 frustration tasks
Combined classification dataset contains 825 samples with 70 features, groups by dataset is 825 members long
Combined datasets subjects list is 825 members long, having 85 distinct subjects
Relevant tasks combined dataset contains 314 samples with 70 features, groups by relevant tasks is 314 members long
Combined relevant tasks subjects list is 314 members long, having 85 distinct subjects


### Dimension/Feature reduction

In [ ]:
# PCA
STD_EXPL_RATIO = 0.95

si_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(si_X)
si_X_95p = si_pca_95p.fit_transform(scaled)

n_rows, n_cols = si_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in si_pca_95p.explained_variance_ratio_]
print(f"StressID classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO*100}% of variance in the dataset")
print(f"Features contribution ratio to variance: {ratios}")



ed_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(ed_X)
ed_X_95p = ed_pca_95p.fit_transform(scaled)

n_rows, n_cols = ed_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in ed_pca_95p.explained_variance_ratio_]
print(
    f"\n\nExpData classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO * 100}% of variance in the dataset"
)
print(f"Features contribution ratio to variance: {ratios}")



all_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(all_X)
all_X_95p = all_pca_95p.fit_transform(scaled)

n_rows, n_cols = all_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in all_pca_95p.explained_variance_ratio_]
print(
    f"\n\nAllData classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO * 100}% of variance in the dataset"
)
print(f"Features contribution ratio to variance: {ratios}")

In [3]:
SPLITS = 10
RAN_STATE = 21

In [4]:
# RFECV for StressID
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(si_X, si_bclass_labels)
features_mask = selector.support_
si_X_rfe = si_X.loc[:, features_mask]

features_scores = { "scores": [], "features": [] }
for score, feat in zip(selector.estimator_.feature_importances_, si_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for StressID")
display(features_scores_df)

Selected features scores for StressID


,scores,features
1,0.111852,pNN20
2,0.102654,ULF
3,0.078333,sampEn
4,0.049976,pNN50
5,0.049237,mean_scl
...,...,...
62,0.002662,ku_eda
63,0.002612,q3_ecg
64,0.002235,max_ecg
65,0.001968,rLF


In [5]:
# RFECV for ExpData
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(ed_X, ed_bclass_labels)
features_mask = selector.support_
ed_X_rfe = ed_X.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, ed_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for ExpData")
display(features_scores_df)

Selected features scores for ExpData


,scores,features
1,0.043895,sd_scl
2,0.043848,scl_slope
3,0.039287,pNN50
4,0.036797,pNN20
5,0.035851,minNN
6,0.032818,median_ecg
7,0.031811,ku_ecg
8,0.029196,modeHR
9,0.028810,minHR
10,0.028653,mean_eda


In [6]:
# RFECV for AllData
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(all_X, all_y)
features_mask = selector.support_
all_X_rfe = all_X.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, all_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for AllData")
display(features_scores_df)

Selected features scores for AllData


,scores,features
1,0.138323,ULF
2,0.127680,pNN20
3,0.067581,sampEn
4,0.052307,median_eda
5,0.051572,pNN50
6,0.050610,mean_eda
7,0.043992,mean_scl
8,0.030880,dynrange
9,0.026595,min_scl
10,0.025919,min_eda


In [7]:
# RFECV for Relevant Tasks in AllData
estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(rel_X, rel_y)
features_mask = selector.support_
rel_X_rfe = rel_X.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, rel_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for Relevant Tasks in AllData")
display(features_scores_df)

Selected features scores for Relevant Tasks in AllData


,scores,features
1,1.0,pNN20


### t-SNE

In [8]:
config = {
    'responsive': False,
    'toImageButtonOptions': {
        'format': 'png', # one of png, svg, jpeg, webp
        'height': 1200,
        'width': 1500,
        'scale': 1 # Multiply title/legend/axis/canvas sizes by this factor
  }
}

In [ ]:
# Divergence analysis on StressID
perplexity = np.arange(30, 390, 30)
divergence = []
si_Ncomp = 2

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(si_X)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=2, Perp=300 np.arange(15, 330, 15)
si_Ncomp = 2
si_Perp = 300
BY_CLASS = False
colors = si_bclass_labels if BY_CLASS else si_groups_list

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X)
display(si_tsne.kl_divergence_)

fig = px.scatter(x=si_X_tsne[:,0], y=si_X_tsne[:,1], color=colors, symbol=si_bclass_labels, width=800, height=600)
fig.update_layout(
    title="StressID ManFeats dataset t-SNE (No Feat Selection)",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(si_subject_to_group)), size=n_subjects)
print(subjects_to_show)
subject_mask = [n in subjects_to_show for n in si_groups_list]
filtered_subjects_rows = [str(subject) for subject, flag in zip(si_groups_list, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(si_bclass_labels.values, subject_mask) if flag]

fig_subj = px.scatter(x=si_X_tsne[:, 0][subject_mask], y=si_X_tsne[:, 1][subject_mask], color=filtered_subjects_rows, symbol=filtered_classes_rows, width=800, height=600)
fig_subj.update_layout(
    title=f"StressID ManFeats dataset t-SNE (No Feat Selection) by {n_subjects} Subject",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=12, opacity=0.7),
    selector=dict(mode="markers")
)
fig_subj.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=3, Perp=360 np.arange(30, 390, 30)
si_Ncomp = 3
si_Perp = 360
BY_CLASS = False
colors = si_bclass_labels if BY_CLASS else si_groups_list

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X)
display(si_tsne.kl_divergence_)

fig = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=colors,
    symbol=si_bclass_labels,
    opacity=0.7,
    width=800,
    height=600,
)
fig.update_layout(title="StressID ManFeats dataset t-SNE (No Feat Selection)")
fig.show()

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(si_subject_to_group)), size=n_subjects)
print(subjects_to_show)
subject_mask = [n in subjects_to_show for n in si_groups_list]
filtered_subjects_rows = [str(subject) for subject, flag in zip(si_groups_list, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(si_bclass_labels.values, subject_mask) if flag]

fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0][subject_mask],
    y=si_X_tsne[:, 1][subject_mask],
    z=si_X_tsne[:, 2][subject_mask],
    color=filtered_subjects_rows,
    symbol=filtered_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title=f"StressID ManFeats dataset t-SNE (No Feat Selection) by {n_subjects} Subject")
fig_subj.update_traces(
    marker=dict(size=14, opacity=0.7),
    selector=dict(mode="markers")
)
fig_subj.show()

In [ ]:
# Divergence analysis on StressID
perplexity = np.arange(25, 700, 25)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(si_X_rfe)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()


In [9]:
# t-SNE in StressID with PCA si_X_95p
# Best N-comp=2, Perp=300 np.arange(50, 700, 50)
# Best N-comp=3, Perp from 550 to 700, np.arange(50, 700, 50)

# t-SNE in StressID with RFE si_X_rfe
# Best N-comp=2, Perp=225 onwards np.arange(25, 700, 25)
# Best N-comp=3, Perp=150 onwards np.arange(25, 700, 25)

si_Ncomp = 3
si_Perp = 150
BY_CLASS = False
colors = si_bclass_labels if BY_CLASS else si_groups_list

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(si_X_rfe)
display(si_tsne.kl_divergence_)

#fig = px.scatter(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], color=si_bclass_labels, width=800, height=600)
fig = px.scatter_3d(
    x=si_X_tsne[:, 0],
    y=si_X_tsne[:, 1],
    z=si_X_tsne[:, 2],
    color=colors,
    symbol=si_bclass_labels,
    opacity=0.7, width=800, height=600
)
fig.update_layout(title="StressID ManFeats dataset t-SNE (RFE Selection)")
fig.update_traces(
    marker=dict(size=18, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-StressID-RFE-AllSubjects"
fig.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(si_subject_to_group)), size=n_subjects)
print(subjects_to_show)
subject_mask = [n in subjects_to_show for n in si_groups_list]
filtered_subjects_rows = [str(subject) for subject, flag in zip(si_groups_list, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(si_bclass_labels.values, subject_mask) if flag]

fig_subj = px.scatter_3d(
    x=si_X_tsne[:, 0][subject_mask],
    y=si_X_tsne[:, 1][subject_mask],
    z=si_X_tsne[:, 2][subject_mask],
    color=filtered_subjects_rows,
    symbol=filtered_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title=f"StressID ManFeats dataset t-SNE (RFE Selection) by {n_subjects} Subject")
fig_subj.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-StressID-RFE-4Subjects"
fig_subj.show(config=config)

0.0738561823964119

[28 43 54 49]


In [ ]:
# Divergence analysis on ExpData
perplexity = np.arange(5, 100, 5)
divergence = []
ed_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=ed_Ncomp, perplexity=i, learning_rate=30)
    reduced = model.fit_transform(ed_X_rfe)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()


In [ ]:
# t-SNE in ExpData with RFE
# Best N-comp=2, Perp=50 onwards (perp range(5, 100, 5))
# Best N-comp=3, Perp=35 onwards (perp range(5, 100, 5)) learning_rate = 30
ed_Ncomp = 3
ed_Perp = 50
BY_CLASS = False
colors = ed_bclass_labels if BY_CLASS else ed_groups_list

ed_tsne = TSNE(n_components=ed_Ncomp, perplexity=ed_Perp, random_state=RAN_STATE, learning_rate=30)
ed_X_tsne = ed_tsne.fit_transform(ed_X_rfe)
display(ed_tsne.kl_divergence_)

#fig = px.scatter(x=ed_X_tsne[:,0], y=ed_X_tsne[:,1], color=ed_bclass_labels, width=800, height=600)
fig = px.scatter_3d(
    x=ed_X_tsne[:, 0],
    y=ed_X_tsne[:, 1],
    z=ed_X_tsne[:, 2],
    color=colors,
    symbol=ed_bclass_labels,
    opacity=0.7, width=800, height=600
)
fig.update_layout(
    title="ExpDataset ManFeats dataset t-SNE (RFE Selection)",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.update_traces(
    marker=dict(size=18, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-ExpData-RFE-AllSubjects"
fig.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(ed_subject_to_group)), size=n_subjects)
print(subjects_to_show)
subject_mask = [n in subjects_to_show for n in ed_groups_list]
filtered_subjects_rows = [str(subject) for subject, flag in zip(ed_groups_list, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(ed_bclass_labels.values, subject_mask) if flag]

fig_subj = px.scatter_3d(
    x=ed_X_tsne[:, 0][subject_mask],
    y=ed_X_tsne[:, 1][subject_mask],
    z=ed_X_tsne[:, 2][subject_mask],
    color=filtered_subjects_rows,
    symbol=filtered_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title=f"ExpDataset ManFeats dataset t-SNE (RFE Selection) by {n_subjects} Subject")
fig_subj.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-ExpData-RFE-4Subjects"
fig_subj.show(config=config)


In [ ]:
# t-SNE in StressID
# Best N-comp=3, Perp=10 (perp range(1, 40, 1))
ed_Ncomp = 3
ed_Perp = 10
BY_CLASS = False
colors = ed_bclass_labels if BY_CLASS else ed_groups_list

ed_tsne = TSNE(n_components=ed_Ncomp, perplexity=ed_Perp, random_state=RAN_STATE)
ed_X_tsne = ed_tsne.fit_transform(ed_X)
display(ed_tsne.kl_divergence_)

fig = px.scatter_3d(
    x=ed_X_tsne[:, 0],
    y=ed_X_tsne[:, 1],
    z=ed_X_tsne[:, 2],
    color=colors,
    symbol=ed_bclass_labels,
    opacity=0.7, width=800, height=600
)
fig.update_layout(
    title="ExpDataset ManFeats dataset t-SNE (No feat selection)",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=14, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-ExpData-NoFS-AllSubjects"
fig.show(config=config)

# Picking a small sample of subjects to show
n_subjects = 4
subjects_to_show = np.random.choice(range(len(ed_subject_to_group)), size=n_subjects)
print(subjects_to_show)
subject_mask = [n in subjects_to_show for n in ed_groups_list]
filtered_subjects_rows = [str(subject) for subject, flag in zip(ed_groups_list, subject_mask) if flag]
filtered_classes_rows = [str(label) for label, flag in zip(ed_bclass_labels.values, subject_mask) if flag]

fig_subj = px.scatter_3d(
    x=ed_X_tsne[:, 0][subject_mask],
    y=ed_X_tsne[:, 1][subject_mask],
    z=ed_X_tsne[:, 2][subject_mask],
    color=filtered_subjects_rows,
    symbol=filtered_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title=f"ExpDataset ManFeats dataset t-SNE (No feat selection) by {n_subjects} Subject")
fig_subj.update_traces(
    marker=dict(size=14, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-ExpData-NoFS-4Subjects"
fig_subj.show(config=config)


### Combined datasets analysis 

In [ ]:
# Divergence analysis on AllData
perplexity = np.arange(25, 700, 25)
divergence = []
si_Ncomp = 2

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(all_X_rfe)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

# Best all_X     N-comp=2, Perp=425 (perp range(25, 700, 25))
# Best all_X_rfe N-comp=2, Perp=450 (perp range(25, 700, 25))
# Best all_X     N-comp=3, Perp=325 (perp range(25, 700, 25))
# Best all_X_rfe N-comp=3, Perp=325 (perp range(25, 700, 25))

In [ ]:
# t-SNE in AllData no feature selection
all_Ncomp = 3
all_Perp = 325
BY_CLASS = False

all_tsne = TSNE(n_components=all_Ncomp, perplexity=all_Perp, random_state=RAN_STATE)
all_X_tsne = all_tsne.fit_transform(all_X)
display(all_tsne.kl_divergence_)

fig = px.scatter_3d(
    x=all_X_tsne[:, 0],
    y=all_X_tsne[:, 1],
    z=all_X_tsne[:, 2],
    color=all_groups.astype(str),
    opacity=0.7, width=800, height=600
)
fig.update_layout(title="Combined ManFeats two-datasets t-SNE (No feat selection)")
fig.update_traces(
    marker=dict(size=18, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-Combined-NoFS-AllSubjects"
fig.show(config=config)

all_subject_mask = si_subject_mask + ed_subject_mask
fig_subj = px.scatter_3d(
    x=all_X_tsne[:, 0][all_subject_mask],
    y=all_X_tsne[:, 1][all_subject_mask],
    z=all_X_tsne[:, 2][all_subject_mask],
    color=all_subsample_subjects_rows,
    symbol=all_subsample_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
fig_subj.update_layout(title=f"Combined ManFeats two-datasets t-SNE (No feat selection) by {len(set(all_subsample_subjects_rows))} subjects")
fig_subj.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = f"tSNE-Combined-NoFS-{len(set(all_subsample_subjects_rows))}Subjects"
fig_subj.show(config=config)

In [ ]:
# t-SNE in AllData with RFE
all_Ncomp = 3
all_Perp = 325
BY_CLASS = False

all_tsne = TSNE(n_components=all_Ncomp, perplexity=all_Perp, random_state=RAN_STATE)
all_X_tsne = all_tsne.fit_transform(all_X_rfe)
display(all_tsne.kl_divergence_)

fig = px.scatter_3d(
    x=all_X_tsne[:, 0],
    y=all_X_tsne[:, 1],
    z=all_X_tsne[:, 2],
    color=all_groups.astype(str),
    opacity=0.7, width=800, height=600
)
fig.update_layout(title="Combined ManFeats dataset t-SNE (RFE)")
fig.update_traces(
    marker=dict(size=18, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = "tSNE-Combined-RFE-AllSubjects"
fig.show(config=config)

all_subject_mask = si_subject_mask + ed_subject_mask
fig_subj = px.scatter_3d(
    x=all_X_tsne[:, 0][all_subject_mask],
    y=all_X_tsne[:, 1][all_subject_mask],
    z=all_X_tsne[:, 2][all_subject_mask],
    color=all_subsample_subjects_rows,
    symbol=all_subsample_classes_rows,
    opacity=0.7,
    width=800,
    height=600,
)
n_subjects = len(set(all_subsample_subjects_rows))
fig_subj.update_layout(title=f"Combined ManFeats dataset t-SNE (RFE) by {n_subjects} Subject")
fig_subj.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = f"tSNE-Combined-RFE-{n_subjects}Subjects"
fig_subj.show(config=config)

In [ ]:
# Divergence analysis on Relevant Tasks in AllData
perplexity = np.arange(15, 300, 15)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(rel_X)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

# Best all_X     N-comp=2, Perp=195 (perp range(25, 700, 25))
# Best all_X     N-comp=3, Perp=195 (perp range(25, 700, 25))

In [ ]:
# t-SNE in Relevant Tasks in AllData without feature selection
rel_Ncomp = 2
rel_Perp = 195

rel_tsne = TSNE(n_components=rel_Ncomp, perplexity=rel_Perp, random_state=RAN_STATE)
rel_X_tsne = rel_tsne.fit_transform(rel_X)
display(rel_tsne.kl_divergence_)

fig = px.scatter(x=rel_X_tsne[:,0], y=rel_X_tsne[:,1], color=rel_groups.astype(str), symbol=rel_y, width=800, height=600)
fig.update_layout(
    title="Relevant Tasks in AllData t-SNE (No Feat Selection)",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = f"tSNE-Combined-RFE-VidTasks-AllSubjects"
fig.show(config=config)

sel_subject_mask: pd.Series = pd.concat([pd.Series(sisel_subject_mask), pd.Series(ed_subject_mask)])
fig_subj = px.scatter(
    x=rel_X_tsne[:, 0][sel_subject_mask],
    y=rel_X_tsne[:, 1][sel_subject_mask],
    color=sel_subsample_subjects_rows.astype(str),
    symbol=sel_subsample_classes_rows,
    width=800,
    height=600
)
fig_subj.update_layout(
    title=f"Relevant Tasks in AllData t-SNE (No Feat Selection) by {len(set(sel_subsample_subjects_rows))} Subjects",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig_subj.update_traces(
    marker=dict(size=22, opacity=0.7),
    selector=dict(mode="markers")
)
config["toImageButtonOptions"]["filename"] = f"tSNE-Combined-RFE-VidTasks-{n_subjects}Subjects"
fig_subj.show(config=config)